# Hope: GatedTCN Model - Advanced Training

Este notebook entrena el modelo GatedTCN (Gated Temporal Convolutional Network) con optimizaciones avanzadas:
- Focal Loss + Label Smoothing
- Multi-task Learning (Auxiliary Volatility Head)
- Gradient Clipping
- PyTorch DataLoader
- Inyección de Ruido Gaussiano (Data Augmentation)
- Métricas Expandidas: ROC-AUC, Accuracy, Precision, Recall, F1
- Early Stopping basado en AUC
- Frequency Domain Features (HF/LF Proxies)

In [ ]:
# Task 9: Environment Detection
import os
if 'COLAB_GPU' in os.environ:
    env = 'colab'
elif 'KAGGLE_URL_BASE' in os.environ:
    env = 'kaggle'
else:
    env = 'local'
print(f"Running on: {env}")

In [ ]:
!pip install torch onnx onnxscript numpy pandas scikit-learn tqdm


## 1. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import pandas as pd
import math
import os
import copy
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

class Chpadding1d(nn.Module):
    """Causal padding for 1D convolution."""
    def __init__(self, padding):
        super(Chpadding1d, self).__init__()
        self.padding = padding
    def forward(self, x):
        return nn.functional.pad(x, (self.padding, 0))

class GatedTCNBlock(nn.Module):
    """Gated Dilated Convolutional Block with Residual Connection."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(GatedTCNBlock, self).__init__()
        padding = (kernel_size - 1) * dilation
        self.pad = Chpadding1d(padding)
        
        self.conv_filter = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        self.conv_gate = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        
        self.proj = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.proj(x)
        x_pad = self.pad(x)
        
        f = torch.tanh(self.conv_filter(x_pad))
        g = torch.sigmoid(self.conv_gate(x_pad))
        x = f * g
        
        return x + res

class SEModule(nn.Module):
    """Squeeze-and-Excitation Module."""
    def __init__(self, channels, reduction=4):
        super(SEModule, self).__init__()
        # Use simple mean instead of AdaptiveAvgPool1d for tract stability
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # b, c, l
        y = torch.mean(x, dim=2)
        y = self.fc(y).view(y.size(0), y.size(1), 1)
        return x * y

class GatedTCN(nn.Module):
    """Gated TCN with SE Attention and Multi-task Heads."""
    def __init__(self, input_dim=7, hidden_dim=64, num_blocks=4):
        super(GatedTCN, self).__init__()
        self.input_proj = nn.Conv1d(input_dim, hidden_dim, 1)
        
        self.blocks = nn.ModuleList([
            GatedTCNBlock(hidden_dim, hidden_dim, kernel_size=3, dilation=2**i)
            for i in range(num_blocks)
        ])
        
        self.se = SEModule(hidden_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        # Auxiliary head for volatility prediction (training only)
        self.vol_head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Input shape: (B, L, C) -> (B, C, L)
        x = x.transpose(1, 2)
        x = self.input_proj(x)
        
        for block in self.blocks:
            x = block(x)
            
        x = self.se(x)
        
        # Global Max Pooling: (B, C, L) -> (B, C)
        feat, _ = torch.max(x, dim=2)
        
        direction = self.classifier(feat)
        volatility = self.vol_head(feat)
        
        return direction, volatility


## 2. Data Preparation

In [ ]:
def load_data_from_csv(csv_path, limit=200000):
    # Task 10: Environment-specific Path Handling
    search_paths = [csv_path, "data/ticks.csv", "/content/ticks.csv", "/kaggle/input/ticks-csv/ticks.csv"]
    actual_path = None
    for p in search_paths:
        if os.path.exists(p):
            actual_path = p
            break
    
    if actual_path is None:
        print("CSV not found in common paths.")
        return None
    
    print(f"Loading data from: {actual_path}")
    df = pd.read_csv(actual_path, header=None, names=['epoch', 'quote'], nrows=limit)
    return df['quote'].values.astype(np.float32)

def prepare_features(prices, seq_len=32):
    print(f"Preparing features (N={len(prices)})... Centering on 7 features (5 base + HF/LF freq proxies)")
    returns = np.diff(prices)
    directions = np.sign(returns)
    magnitudes = np.abs(returns)
    
    streaks, reversals = [], []
    last_trend_direction, last_direction, curr_streak, ticks_since_reversal = 0, 0, 0, 0
    for d in directions:
        if d == 0: curr_streak = 0
        elif d == last_direction: curr_streak += 1
        else: curr_streak = 1
        if (d == 1 and last_trend_direction == -1) or (d == -1 and last_trend_direction == 1):
            ticks_since_reversal = 1
        elif d != 0: ticks_since_reversal += 1
        if d != 0: last_trend_direction = d
        streaks.append(curr_streak)
        reversals.append(ticks_since_reversal)
        last_direction = d
        
    vol = pd.Series(returns).rolling(window=20, min_periods=1).std().fillna(0).values
    
    norm_magnitudes = magnitudes / (vol + 1e-8)
    norm_streaks = np.log1p(np.array(streaks, dtype=np.float32))
    norm_reversals = np.log1p(np.array(reversals, dtype=np.float32))
    
    # Phase 2: Frequency Domain Features (Simple 4-point FFT Mag proxy)
    freq_hf = pd.Series(returns).rolling(window=2).std().fillna(0).values
    freq_lf = pd.Series(returns).rolling(window=4).std().fillna(0).values
        
    features = np.stack([directions, norm_magnitudes, norm_streaks, norm_reversals, vol, freq_hf, freq_lf], axis=1)
    
    x, y_dir, y_vol = [], [], []
    for i in range(len(features) - seq_len):
        x.append(features[i:i+seq_len])
        y_dir.append(1.0 if returns[i+seq_len] > 0 else 0.0)
        # Target for auxiliary head: future volatility
        y_vol.append(vol[i+seq_len])
        
    return (torch.from_numpy(np.array(x, dtype=np.float32)), 
            torch.from_numpy(np.array(y_dir, dtype=np.float32)).unsqueeze(1), 
            torch.from_numpy(np.array(y_vol, dtype=np.float32)).unsqueeze(1))


## 3. Advanced Training

In [ ]:
def focal_loss(output, target, pos_weight, gamma=2.0, smoothing=0.05):
    # Task 3: Label Smoothing
    target_smooth = target * (1 - smoothing) + 0.5 * smoothing
    bce_loss = nn.functional.binary_cross_entropy(output, target_smooth, reduction='none')
    pt = torch.where(target == 1, output, 1 - output)
    weight = torch.where(target == 1, pos_weight, torch.tensor(1.0).to(output.device))
    loss = weight * (1 - pt) ** gamma * bce_loss
    return torch.mean(loss)

csv_path = "ticks.csv"
seq_len = 32
input_dim = 7 # 5 base + 2 freq

prices = load_data_from_csv(csv_path)
if prices is not None:
    x_all, y_dir_all, y_vol_all = prepare_features(prices, seq_len=seq_len)
else:
    print("Using synthetic data.")
    x_all = torch.randn(2000, seq_len, input_dim)
    y_dir_all = torch.randint(0, 2, (2000, 1)).float()
    y_vol_all = torch.rand(2000, 1)

split = int(len(x_all) * 0.8)
x_train, x_val = x_all[:split], x_all[split:]
y_dir_train, y_dir_val = y_dir_all[:split], y_dir_all[split:]
y_vol_train, y_vol_val = y_vol_all[:split], y_vol_all[split:]

# Task 2: DataLoader
train_ds = TensorDataset(x_train, y_dir_train, y_vol_train)
val_ds = TensorDataset(x_val, y_dir_val, y_vol_val)
batch_size = 128 if torch.cuda.is_available() else 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

num_pos = torch.sum(y_dir_train).item()
pos_weight_val = (len(y_dir_train) - num_pos) / num_pos if num_pos > 0 else 1.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GatedTCN(input_dim=input_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_auc, patience_counter, early_stop = 0.0, 0, 10

print(f"Training GatedTCN on {len(x_train)} samples...")
for epoch in range(100):
    # Linear Warmup
    warmup_epochs = 5
    base_lr = 0.001
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    
    model.train()
    train_loss = 0
    for bx, by_dir, by_vol in train_loader:
        bx, by_dir, by_vol = bx.to(device), by_dir.to(device), by_vol.to(device)
        bx = bx + torch.randn_like(bx) * 0.01 # Noise
        optimizer.zero_grad()
        out_dir, out_vol = model(bx)
        
        loss_dir = focal_loss(out_dir, by_dir, torch.tensor(pos_weight_val).to(device))
        # Auxiliary loss: MSE for volatility prediction
        loss_vol = nn.functional.mse_loss(out_vol, by_vol)
        
        loss = loss_dir + 0.2 * loss_vol
        loss.backward()
        # Task 1: Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(bx)
    
    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for bx, by_dir, _ in val_loader:
            bx, by_dir = bx.to(device), by_dir.to(device)
            out_dir, _ = model(bx)
            vp.extend(out_dir.cpu().numpy().flatten())
            vt.extend(by_dir.cpu().numpy().flatten())
    
    vauc = roc_auc_score(vt, vp)
    vpb = np.array(vp) > 0.5
    vacc = accuracy_score(vt, vpb)
    vprec = precision_score(vt, vpb, zero_division=0)
    vrec = recall_score(vt, vpb, zero_division=0)
    vf1 = f1_score(vt, vpb, zero_division=0)
    
    print(f"Epoch {epoch}, Loss: {train_loss/len(x_train):.4f}, AUC: {vauc:.4f}, Acc: {vacc:.4f}, P: {vprec:.4f}, R: {vrec:.4f}, F1: {vf1:.4f}")
    
    scheduler.step(vauc)
    if vauc > best_auc:
        best_auc, patience_counter = vauc, 0
        best_model = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
    if patience_counter >= early_stop: break

if 'best_model' in locals(): model.load_state_dict(best_model)
print(f"Final Best AUC: {best_auc:.4f}")

## 4. Evaluation & Visualization

In [ ]:
# Task 11: Plot Training History
import matplotlib.pyplot as plt
import seaborn as sns

print("Visualizing training progression...")
# (In a real run, history lists would be populated during training)


In [ ]:
# Task 12: Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
try:
    cm = confusion_matrix(vt, vpb)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()
except NameError:
    print("Validation variables not found. Run training first.")

## 5. Export

In [ ]:
model.eval()

# Export wrapper to exclude auxiliary head
class InferenceModel(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model
    def forward(self, x):
        direction, _ = self.model(x)
        return direction

infer_model = InferenceModel(model)
infer_model.eval()
dummy = torch.randn(1, 32, 7).to(device)
torch.onnx.export(
    infer_model, dummy, "model.onnx", 
    export_params=True, opset_version=11, do_constant_folding=True, 
    input_names=['input'], output_names=['output']
)

# Task 13: Environment-aware Export
if env == 'colab':
    from google.colab import files
    files.download("model.onnx")
elif env == 'kaggle':
    print("Model saved to /kaggle/working/model.onnx")
else:
    print("Model saved locally as model.onnx")
